# 05 LILAC Training Visualizer

This notebook inspects the learned LILAC geometry after running `02_lilac_train_sh5.ipynb`.
Run the cells in order:

1. Dataset and model load
2. Dataset summary
3. Latent space viewer
4. Basis vector field
5. Action reconstruction
6. Runtime-style end-effector motion plus language timeline


In [ ]:
%matplotlib inline
%run ../package/init_project.py

import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
import torch

from controller import apply_latent_alignment, load_latent_alignment
from language import CanonicalLanguageDataset, CanonicalLanguageIndex
from lilac_model import LILACModel
from paths import CANONICAL_LANGUAGE_DATASET, CANONICAL_LANGUAGE_INDEX, DATA_DIR, PROJECT_DIR, RUN_DIR, TRAINING_ARRAYS

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 10


## 1. Load Dataset And Model

`TRAINING_ARRAYS` is generated by notebook 02. `RUN_DIR` is the single LILAC run directory.


In [ ]:
ARRAYS_PATH = TRAINING_ARRAYS
if not ARRAYS_PATH.exists():
    raise FileNotFoundError("No training arrays found. Run 02_lilac_train_sh5.ipynb first.")

payload = np.load(ARRAYS_PATH, allow_pickle=True)
states = np.asarray(payload["states"], dtype=np.float32)
actions = np.asarray(payload["actions"], dtype=np.float32)
utterances = np.asarray([str(u) for u in payload["utterances"].tolist()], dtype=object)
alphas = np.asarray(payload["alphas"], dtype=np.float32)
latent_z = np.asarray(payload["latent_z"], dtype=np.float32) if "latent_z" in payload else np.full((len(states), 2), np.nan, dtype=np.float32)
episode_ids = np.asarray([str(e) for e in payload["episode_ids"].tolist()], dtype=object) if "episode_ids" in payload else np.asarray(["episode"] * len(states), dtype=object)

language_dataset = CanonicalLanguageDataset.load(CANONICAL_LANGUAGE_DATASET)
model = None
model_config = None
language_index = None
latent_alignment = None

if (RUN_DIR / "model.pt").exists():
    model, model_config = LILACModel.load_bundle(RUN_DIR)
    language_index = CanonicalLanguageIndex.load(RUN_DIR / "language_index.npz")
    if (RUN_DIR / "latent_alignment.npz").exists():
        latent_alignment = load_latent_alignment(RUN_DIR / "latent_alignment.npz")
    print("[model] loaded:", RUN_DIR)
else:
    if CANONICAL_LANGUAGE_INDEX.exists():
        language_index = CanonicalLanguageIndex.load(CANONICAL_LANGUAGE_INDEX)
    print("[model] missing model.pt in:", RUN_DIR)
    print("[model] run 02_lilac_train_sh5.ipynb before model-dependent cells.")

print("Arrays:", ARRAYS_PATH)
print("Run dir:", RUN_DIR)
print("states", states.shape, "actions", actions.shape, "latent_z", latent_z.shape)


In [ ]:
def require_model():
    if model is None:
        raise FileNotFoundError("No model.pt loaded. Run 02_lilac_train_sh5.ipynb, then rerun cells 1-6 here.")
    if language_index is None:
        raise FileNotFoundError("No language_index.npz loaded for the run.")


def model_device():
    require_model()
    return next(model.parameters()).device


def short_label(text, max_len=34):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 3] + "..."


def utterance_colors(texts):
    unique = sorted(set(str(t) for t in texts))
    cmap = plt.get_cmap("tab10" if len(unique) <= 10 else "tab20")
    return {utt: cmap(i % cmap.N) for i, utt in enumerate(unique)}


def sample_balanced_by_utterance(max_per_utterance=1000, seed=7):
    rng = np.random.default_rng(seed)
    idxs = []
    for utt in sorted(set(utterances)):
        group = np.flatnonzero(utterances == utt)
        if len(group) > max_per_utterance:
            group = rng.choice(group, size=max_per_utterance, replace=False)
        idxs.extend(group.tolist())
    return np.asarray(sorted(idxs), dtype=np.int64)


def language_embeddings_for(texts):
    if language_index is None:
        raise FileNotFoundError("language_index is required for model visualization.")
    return np.asarray([language_index.lookup(text)["embedding"] for text in texts], dtype=np.float32)


def align_z_batch(z_values):
    z_values = np.asarray(z_values, dtype=np.float32).reshape(-1, 2)
    if latent_alignment is None:
        return z_values.copy()
    return np.asarray([apply_latent_alignment(z, latent_alignment) for z in z_values], dtype=np.float32)


def encode_action_latents(idxs, batch_size=512):
    require_model()
    device = model_device()
    out = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(idxs), batch_size):
            b = idxs[start:start + batch_size]
            state_t = torch.as_tensor(states[b], dtype=torch.float32, device=device)
            lang_t = torch.as_tensor(language_embeddings_for(utterances[b]), dtype=torch.float32, device=device)
            alpha_t = torch.as_tensor(alphas[b], dtype=torch.float32, device=device)
            action_t = torch.as_tensor(actions[b], dtype=torch.float32, device=device)
            z_t = model.encode_action_latent(state_t, lang_t, alpha_t, action_t)
            out.append(z_t.detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)


def reconstruct_actions(idxs, batch_size=512):
    require_model()
    device = model_device()
    out = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(idxs), batch_size):
            b = idxs[start:start + batch_size]
            state_t = torch.as_tensor(states[b], dtype=torch.float32, device=device)
            lang_t = torch.as_tensor(language_embeddings_for(utterances[b]), dtype=torch.float32, device=device)
            alpha_t = torch.as_tensor(alphas[b], dtype=torch.float32, device=device)
            action_t = torch.as_tensor(actions[b], dtype=torch.float32, device=device)
            pred_t = model.forward(state_t, lang_t, alpha_t, action_t)
            out.append(pred_t.detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)


def decode_actions_from_z(idxs, z_model, batch_size=512):
    require_model()
    device = model_device()
    z_model = np.asarray(z_model, dtype=np.float32).reshape(len(idxs), 2)
    out = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(idxs), batch_size):
            b = idxs[start:start + batch_size]
            z_b = z_model[start:start + batch_size]
            state_t = torch.as_tensor(states[b], dtype=torch.float32, device=device)
            lang_t = torch.as_tensor(language_embeddings_for(utterances[b]), dtype=torch.float32, device=device)
            alpha_t = torch.as_tensor(alphas[b], dtype=torch.float32, device=device)
            z_t = torch.as_tensor(z_b, dtype=torch.float32, device=device)
            pred_t = model.decoder(state_t, lang_t, alpha_t, z_t)
            out.append(pred_t.detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)


def contiguous_episode_slices(ids):
    slices = []
    if len(ids) == 0:
        return slices
    start = 0
    for idx in range(1, len(ids)):
        if ids[idx] != ids[start]:
            slices.append((ids[start], start, idx))
            start = idx
    slices.append((ids[start], start, len(ids)))
    return slices

## 2. Dataset Summary

Use this first to verify what the current arrays actually contain: utterance balance, episode count, action position/rotation magnitude, and joystick recording coverage.

In [ ]:
print("n_samples:", len(states))
print("state_dim:", states.shape[1], "action_dim:", actions.shape[1])
print("n_episodes:", len(set(episode_ids)))

print()
print("Utterance counts:")
for utt in sorted(set(utterances)):
    print("  %-48s %6d" % (short_label(utt, 48), int(np.sum(utterances == utt))))

pos_norm = np.linalg.norm(actions[:, :3], axis=1)
rot_norm = np.linalg.norm(actions[:, 3:], axis=1)
print()
print("Action norm stats, training actions are usually normalized directions:")
print("  pos norm mean/std/min/max: %.4f %.4f %.4f %.4f" % (pos_norm.mean(), pos_norm.std(), pos_norm.min(), pos_norm.max()))
print("  rot norm mean/std/min/max: %.4f %.4f %.4f %.4f" % (rot_norm.mean(), rot_norm.std(), rot_norm.min(), rot_norm.max()))

valid_z = np.isfinite(latent_z).all(axis=1)
nonzero_z = valid_z & (np.linalg.norm(latent_z, axis=1) > 1e-5)
print()
print("Joystick latent_z coverage:")
print("  finite:", int(valid_z.sum()), "/", len(valid_z))
print("  nonzero:", int(nonzero_z.sum()), "/", len(nonzero_z))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pos_norm, bins=40, alpha=0.8, label="||dxyz||")
axes[0].hist(rot_norm, bins=40, alpha=0.7, label="||drpy||")
axes[0].set_title("Action norm distribution")
axes[0].legend()

counts = [int(np.sum(utterances == utt)) for utt in sorted(set(utterances))]
labels = [short_label(utt, 24) for utt in sorted(set(utterances))]
axes[1].bar(np.arange(len(labels)), counts)
axes[1].set_xticks(np.arange(len(labels)))
axes[1].set_xticklabels(labels, rotation=35, ha="right")
axes[1].set_title("Samples per utterance")
axes[1].set_ylabel("frames")
plt.tight_layout()

## 3. Latent Space Viewer

This plots the learned 2D latent `z = compressor(state, language, alpha, action)` and compares it against the recorded joystick latent. If `pour water` does not form a usable region or is heavily mixed with unrelated actions, the correction dataset is probably too weak or inconsistent.

In [ ]:
require_model()

LATENT_SAMPLE_PER_UTTERANCE = 1200
idxs_latent = sample_balanced_by_utterance(max_per_utterance=LATENT_SAMPLE_PER_UTTERANCE, seed=11)
encoded_z = encode_action_latents(idxs_latent)
joy_raw = latent_z[idxs_latent]
joy_valid = np.isfinite(joy_raw).all(axis=1)
joy_aligned = align_z_batch(np.where(joy_valid[:, None], joy_raw, 0.0))
colors = utterance_colors(utterances[idxs_latent])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for utt, color in colors.items():
    m = utterances[idxs_latent] == utt
    axes[0].scatter(encoded_z[m, 0], encoded_z[m, 1], s=8, alpha=0.55, color=color, label=short_label(utt, 24))
axes[0].axhline(0, color="k", linewidth=0.8)
axes[0].axvline(0, color="k", linewidth=0.8)
axes[0].set_title("Model-encoded latent z")
axes[0].set_xlabel("z1")
axes[0].set_ylabel("z2")
axes[0].legend(loc="best", fontsize=8)

for utt, color in colors.items():
    m = (utterances[idxs_latent] == utt) & joy_valid
    if np.any(m):
        axes[1].scatter(joy_aligned[m, 0], joy_aligned[m, 1], s=8, alpha=0.55, color=color, label=short_label(utt, 24))
axes[1].axhline(0, color="k", linewidth=0.8)
axes[1].axvline(0, color="k", linewidth=0.8)
axes[1].set_title("Recorded joystick z after latent alignment" if latent_alignment is not None else "Recorded joystick z")
axes[1].set_xlabel("z1")
axes[1].set_ylabel("z2")
axes[1].legend(loc="best", fontsize=8)
plt.tight_layout()

## 4. Basis Vector Field

For the same reference states, this compares each canonical utterance's learned basis `B in R^(6x2)`. The decoder output is `action_6d = B @ z`. For `pour water`, the rotation basis magnitude should be meaningfully visible if the model learned pouring as an orientation correction.

In [ ]:
require_model()

BASIS_STATE_COUNT = min(250, len(states))
BASIS_UTTERANCES = [entry.text for entry in language_dataset.entries]
rng = np.random.default_rng(21)
state_idxs = np.sort(rng.choice(np.arange(len(states)), size=BASIS_STATE_COUNT, replace=False))
state_ref = states[state_idxs]

def bases_for_utterance(utterance_text):
    entry = language_dataset.get(utterance_text)
    emb = language_index.lookup(entry.id)["embedding"]
    device = model_device()
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(state_ref), 512):
            s = state_ref[start:start + 512]
            state_t = torch.as_tensor(s, dtype=torch.float32, device=device)
            lang_t = torch.as_tensor(np.repeat(emb[None, :], len(s), axis=0), dtype=torch.float32, device=device)
            alpha_t = torch.as_tensor(np.full((len(s),), entry.alpha, dtype=np.float32), device=device)
            b_t = model.bases(state_t, lang_t, alpha_t)
            out.append(b_t.detach().cpu().numpy())
    return np.concatenate(out, axis=0)

basis_rows = []
mean_bases = {}
for utt in BASIS_UTTERANCES:
    B = bases_for_utterance(utt)
    B_mean = B.mean(axis=0)
    mean_bases[utt] = B_mean
    basis_rows.append({
        "utterance": utt,
        "b1_pos": float(np.linalg.norm(B[:, :3, 0], axis=1).mean()),
        "b1_rot": float(np.linalg.norm(B[:, 3:, 0], axis=1).mean()),
        "b2_pos": float(np.linalg.norm(B[:, :3, 1], axis=1).mean()),
        "b2_rot": float(np.linalg.norm(B[:, 3:, 1], axis=1).mean()),
    })

print("Mean basis magnitudes over %d shared reference states:" % BASIS_STATE_COUNT)
print("%-38s %9s %9s %9s %9s" % ("utterance", "b1_pos", "b1_rot", "b2_pos", "b2_rot"))
for row in basis_rows:
    print("%-38s %9.4f %9.4f %9.4f %9.4f" % (
        short_label(row["utterance"], 38), row["b1_pos"], row["b1_rot"], row["b2_pos"], row["b2_rot"]
    ))

colors = utterance_colors(BASIS_UTTERANCES)
fig = plt.figure(figsize=(15, 6))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
for utt in BASIS_UTTERANCES:
    color = colors[utt]
    Bm = mean_bases[utt]
    ax3d.quiver(0, 0, 0, Bm[0, 0], Bm[1, 0], Bm[2, 0], color=color, linewidth=2, arrow_length_ratio=0.15)
    ax3d.quiver(0, 0, 0, Bm[0, 1], Bm[1, 1], Bm[2, 1], color=color, linewidth=1, linestyle="dashed", arrow_length_ratio=0.15)
ax3d.set_title("Mean translation basis arrows: solid=b1, dashed=b2")
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.legend([plt.Line2D([0], [0], color=colors[u], lw=2) for u in BASIS_UTTERANCES], [short_label(u, 22) for u in BASIS_UTTERANCES], fontsize=8, loc="upper left")

ax = fig.add_subplot(1, 2, 2)
x = np.arange(len(BASIS_UTTERANCES))
width = 0.18
ax.bar(x - 1.5 * width, [r["b1_pos"] for r in basis_rows], width, label="b1 pos")
ax.bar(x - 0.5 * width, [r["b1_rot"] for r in basis_rows], width, label="b1 rot")
ax.bar(x + 0.5 * width, [r["b2_pos"] for r in basis_rows], width, label="b2 pos")
ax.bar(x + 1.5 * width, [r["b2_rot"] for r in basis_rows], width, label="b2 rot")
ax.set_xticks(x)
ax.set_xticklabels([short_label(u, 18) for u in BASIS_UTTERANCES], rotation=35, ha="right")
ax.set_title("Basis translation vs rotation magnitude")
ax.legend()
plt.tight_layout()

## 5. Action Reconstruction

This compares the normalized training action with the model reconstruction. High `pour water` rotation error means the model is not reproducing the pouring motion from the recorded state/language/action manifold.

In [ ]:
require_model()

RECON_SAMPLE_PER_UTTERANCE = 2000
idxs_recon = sample_balanced_by_utterance(max_per_utterance=RECON_SAMPLE_PER_UTTERANCE, seed=31)
pred_actions = reconstruct_actions(idxs_recon)
true_actions = actions[idxs_recon]
recon_utts = utterances[idxs_recon]
err = pred_actions - true_actions

print("Reconstruction MSE by utterance:")
print("%-38s %12s %12s %12s" % ("utterance", "all", "pos", "rot"))
rows = []
for utt in sorted(set(recon_utts)):
    m = recon_utts == utt
    mse_all = float(np.mean(err[m] ** 2))
    mse_pos = float(np.mean(err[m, :3] ** 2))
    mse_rot = float(np.mean(err[m, 3:] ** 2))
    rows.append((utt, mse_all, mse_pos, mse_rot))
    print("%-38s %12.6f %12.6f %12.6f" % (short_label(utt, 38), mse_all, mse_pos, mse_rot))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(rows))
axes[0].bar(x - 0.18, [r[2] for r in rows], 0.36, label="pos MSE")
axes[0].bar(x + 0.18, [r[3] for r in rows], 0.36, label="rot MSE")
axes[0].set_xticks(x)
axes[0].set_xticklabels([short_label(r[0], 22) for r in rows], rotation=35, ha="right")
axes[0].set_title("Reconstruction error by utterance")
axes[0].legend()

for dim, name in enumerate(["dx", "dy", "dz", "droll", "dpitch", "dyaw"]):
    axes[1].scatter(true_actions[:, dim], pred_actions[:, dim], s=5, alpha=0.25, label=name)
lim = max(float(np.max(np.abs(true_actions))), float(np.max(np.abs(pred_actions))), 1e-6)
axes[1].plot([-lim, lim], [-lim, lim], color="k", linewidth=1)
axes[1].set_xlim(-lim, lim)
axes[1].set_ylim(-lim, lim)
axes[1].set_title("Predicted vs true action components")
axes[1].set_xlabel("true")
axes[1].set_ylabel("predicted")
axes[1].legend(ncol=2, fontsize=8)
plt.tight_layout()

In [ ]:
# Per-episode action trace. Set EPISODE_ID_TO_PLOT manually to inspect a specific demonstration.
EPISODE_ID_TO_PLOT = None
EPISODE_UTTERANCE_FILTER = "pour water"  # set to None for the longest episode overall

slices = contiguous_episode_slices(episode_ids)
if EPISODE_ID_TO_PLOT is not None:
    selected = [s for s in slices if s[0] == EPISODE_ID_TO_PLOT]
else:
    selected = slices
    if EPISODE_UTTERANCE_FILTER is not None:
        selected = [s for s in slices if np.any(utterances[s[1]:s[2]] == EPISODE_UTTERANCE_FILTER)] or slices
selected_id, start, end = max(selected, key=lambda s: s[2] - s[1])
idxs_ep = np.arange(start, end, dtype=np.int64)

pred_ep = reconstruct_actions(idxs_ep)
true_ep = actions[idxs_ep]
t = np.arange(len(idxs_ep))

fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)
for ax, dim, name in zip(axes.ravel(), range(6), ["dx", "dy", "dz", "droll", "dpitch", "dyaw"]):
    ax.plot(t, true_ep[:, dim], label="true", linewidth=1.3)
    ax.plot(t, pred_ep[:, dim], label="recon", linewidth=1.1, alpha=0.85)
    ax.set_title(name)
    ax.legend(fontsize=8)
fig.suptitle("Action reconstruction trace: %s" % selected_id)
plt.tight_layout()

## 6. Runtime-Style EE Motion + Language Timeline

This combines the two runtime diagnostics: end-effector motion and active language timeline. It replays one recorded episode, then decodes runtime-style actions from the recorded joystick `z` through the trained decoder. This is the closest offline view to what notebook 04 will do at deployment time.

In [ ]:
require_model()

EPISODE_ID_TO_PLOT = None
EPISODE_UTTERANCE_FILTER = "pour water"  # set to None to pick the longest episode overall
RUNTIME_ACTION_POS_SCALE = 0.01
RUNTIME_ACTION_ROT_SCALE = 0.0175
MAX_ARROWS = 50

slices = contiguous_episode_slices(episode_ids)
if EPISODE_ID_TO_PLOT is not None:
    candidates = [s for s in slices if s[0] == EPISODE_ID_TO_PLOT]
    if not candidates:
        raise ValueError("Unknown EPISODE_ID_TO_PLOT: %s" % EPISODE_ID_TO_PLOT)
else:
    candidates = slices
    if EPISODE_UTTERANCE_FILTER is not None:
        candidates = [s for s in slices if np.any(utterances[s[1]:s[2]] == EPISODE_UTTERANCE_FILTER)] or slices
selected_id, start, end = max(candidates, key=lambda s: s[2] - s[1])
idxs_rt = np.arange(start, end, dtype=np.int64)

xyz = states[idxs_rt, 7:10]
rpy = states[idxs_rt, 10:13]
utt_rt = utterances[idxs_rt]
z_raw = latent_z[idxs_rt]
z_valid = np.isfinite(z_raw).all(axis=1)
if not np.any(z_valid):
    z_used_raw = encode_action_latents(idxs_rt)
    z_source = "model-encoded z because recorded joystick_z is unavailable"
else:
    z_used_raw = np.where(z_valid[:, None], z_raw, 0.0)
    z_source = "recorded joystick_z"
z_model = align_z_batch(z_used_raw)
raw_decoded = decode_actions_from_z(idxs_rt, z_model)
runtime_actions = raw_decoded.copy()
runtime_actions[:, :3] *= RUNTIME_ACTION_POS_SCALE
runtime_actions[:, 3:] *= RUNTIME_ACTION_ROT_SCALE
runtime_xyz = np.vstack([xyz[0], xyz[0] + np.cumsum(runtime_actions[:, :3], axis=0)])

obj_flat = states[idxs_rt[0], 13:]
obj_points = obj_flat.reshape(-1, 3) if len(obj_flat) >= 3 and len(obj_flat) % 3 == 0 else np.zeros((0, 3))

colors = utterance_colors(utt_rt)
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 2, height_ratios=[2.0, 1.0, 1.0], hspace=0.35, wspace=0.25)

ax3d = fig.add_subplot(gs[0, 0], projection="3d")
for utt, color in colors.items():
    m = utt_rt == utt
    ax3d.scatter(xyz[m, 0], xyz[m, 1], xyz[m, 2], s=10, alpha=0.75, color=color, label=short_label(utt, 24))
ax3d.plot(runtime_xyz[:, 0], runtime_xyz[:, 1], runtime_xyz[:, 2], color="black", linewidth=1.5, label="decoded rollout")
if len(obj_points):
    ax3d.scatter(obj_points[:, 0], obj_points[:, 1], obj_points[:, 2], s=70, marker="x", color="red", label="object_state")
arrow_step = max(1, len(idxs_rt) // MAX_ARROWS)
arrow_base = xyz[::arrow_step]
arrow_vec = runtime_actions[::arrow_step, :3]
ax3d.quiver(arrow_base[:, 0], arrow_base[:, 1], arrow_base[:, 2], arrow_vec[:, 0], arrow_vec[:, 1], arrow_vec[:, 2], length=1.0, normalize=False, color="black", alpha=0.55)
ax3d.set_title("EE trajectory and decoded runtime dxyz arrows")
ax3d.set_xlabel("x")
ax3d.set_ylabel("y")
ax3d.set_zlabel("z")
ax3d.legend(fontsize=8)

ax_pose = fig.add_subplot(gs[0, 1])
t = np.arange(len(idxs_rt))
ax_pose.plot(t, xyz[:, 0], label="x")
ax_pose.plot(t, xyz[:, 1], label="y")
ax_pose.plot(t, xyz[:, 2], label="z")
ax_pose_r = ax_pose.twinx()
ax_pose_r.plot(t, rpy[:, 0], "--", alpha=0.7, label="roll")
ax_pose_r.plot(t, rpy[:, 1], "--", alpha=0.7, label="pitch")
ax_pose_r.plot(t, rpy[:, 2], "--", alpha=0.7, label="yaw")
ax_pose.set_title("Recorded EE pose over episode")
ax_pose.set_xlabel("frame")
ax_pose.set_ylabel("position")
ax_pose_r.set_ylabel("rpy rad")
lines, labels = ax_pose.get_legend_handles_labels()
lines_r, labels_r = ax_pose_r.get_legend_handles_labels()
ax_pose.legend(lines + lines_r, labels + labels_r, fontsize=8, ncol=3)

ax_lang = fig.add_subplot(gs[1, :])
unique_utts = sorted(set(utt_rt))
lane = {utt: i for i, utt in enumerate(unique_utts)}
for utt, color in colors.items():
    m = utt_rt == utt
    ax_lang.scatter(t[m], np.full(np.sum(m), lane[utt]), s=12, color=color, label=short_label(utt, 28))
ax_lang.set_yticks(np.arange(len(unique_utts)))
ax_lang.set_yticklabels([short_label(u, 42) for u in unique_utts])
ax_lang.set_title("Active utterance timeline")
ax_lang.set_xlabel("frame")
ax_lang.set_ylim(-0.5, max(0.5, len(unique_utts) - 0.5))

ax_z = fig.add_subplot(gs[2, 0])
ax_z.plot(t, z_used_raw[:, 0], label="z raw 1", alpha=0.8)
ax_z.plot(t, z_used_raw[:, 1], label="z raw 2", alpha=0.8)
ax_z.plot(t, z_model[:, 0], "--", label="z model 1", alpha=0.8)
ax_z.plot(t, z_model[:, 1], "--", label="z model 2", alpha=0.8)
ax_z.set_title("Latent input: %s" % z_source)
ax_z.set_xlabel("frame")
ax_z.legend(fontsize=8)

ax_act = fig.add_subplot(gs[2, 1])
dxyz_norm = np.linalg.norm(runtime_actions[:, :3], axis=1)
drpy_norm = np.linalg.norm(runtime_actions[:, 3:], axis=1)
ax_act.plot(t, dxyz_norm, label="||runtime dxyz||")
ax_act.plot(t, drpy_norm, label="||runtime drpy||")
ax_act.plot(t, raw_decoded[:, 3], "--", alpha=0.6, label="raw droll")
ax_act.plot(t, raw_decoded[:, 4], "--", alpha=0.6, label="raw dpitch")
ax_act.plot(t, raw_decoded[:, 5], "--", alpha=0.6, label="raw dyaw")
ax_act.set_title("Decoded runtime action magnitude")
ax_act.set_xlabel("frame")
ax_act.legend(fontsize=8, ncol=2)

fig.suptitle("Runtime-style replay: %s" % selected_id, y=0.98)
plt.show()

## 7. Raw Dataset EE Trajectory Snapshot

This cell picks one raw `cup_to_bowl/instruction` demonstration and saves a single 3D image of the recorded end-effector path. It does not require `model.pt`; it only reads the collected `.npz` episode.


In [ ]:
from pathlib import Path
import json

RAW_EE_TRAJECTORY_NPZ = None  # set a specific .npz path to override auto-pick
RAW_EE_TRAJECTORY_TASK = "cup_to_bowl"
RAW_EE_TRAJECTORY_EPISODE_TYPE = "instruction"
RAW_EE_TRAJECTORY_QUERY_WORDS = ["cup", "bowl"]
RAW_EE_TRAJECTORY_MAX_ARROWS = 32
RAW_EE_TRAJECTORY_OUT = PROJECT_DIR / "outputs" / "raw_ee_trajectory" / "cup_to_bowl_ee_trajectory.png"


def load_episode_meta(npz_path):
    meta_path = Path(npz_path).with_suffix(".json")
    if meta_path.exists():
        return json.loads(meta_path.read_text(encoding="utf-8"))
    return {}


def pick_raw_ee_episode():
    if RAW_EE_TRAJECTORY_NPZ is not None:
        return Path(RAW_EE_TRAJECTORY_NPZ)

    episode_dir = DATA_DIR / RAW_EE_TRAJECTORY_TASK / RAW_EE_TRAJECTORY_EPISODE_TYPE
    candidates = []
    for npz_path in sorted(episode_dir.glob("*.npz")):
        meta = load_episode_meta(npz_path)
        instruction = str(meta.get("instruction", "")).lower()
        if RAW_EE_TRAJECTORY_QUERY_WORDS:
            if not all(word.lower() in instruction for word in RAW_EE_TRAJECTORY_QUERY_WORDS):
                continue
        data = np.load(npz_path, allow_pickle=True)
        if "ee_pose" not in data:
            continue
        n_frames = int(meta.get("n_frames", len(data["ee_pose"])))
        candidates.append((n_frames, npz_path))

    if not candidates:
        raise FileNotFoundError(
            "No matching raw EE episodes under %s. Set RAW_EE_TRAJECTORY_NPZ manually." % episode_dir
        )
    return max(candidates, key=lambda item: item[0])[1]


def set_axes_equal_3d(ax, *point_sets, margin=0.04):
    valid = []
    for points in point_sets:
        arr = np.asarray(points, dtype=np.float64)
        if arr.size == 0:
            continue
        valid.append(arr.reshape(-1, 3))
    if not valid:
        return
    pts = np.vstack(valid)
    mins = np.nanmin(pts, axis=0)
    maxs = np.nanmax(pts, axis=0)
    center = 0.5 * (mins + maxs)
    radius = 0.5 * float(np.max(maxs - mins)) + float(margin)
    radius = max(radius, 0.05)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


selected_npz = pick_raw_ee_episode()
selected_meta = load_episode_meta(selected_npz)
episode = np.load(selected_npz, allow_pickle=True)
ee_pose_raw = np.asarray(episode["ee_pose"], dtype=np.float64)
xyz_raw = ee_pose_raw[:, :3]

object_points_start = np.zeros((0, 3), dtype=np.float64)
object_points_end = np.zeros((0, 3), dtype=np.float64)
if "object_state" in episode:
    obj = np.asarray(episode["object_state"], dtype=np.float64)
    if obj.ndim == 1 and obj.size >= 3 and obj.size % 3 == 0:
        object_points_start = obj.reshape(-1, 3)
        object_points_end = object_points_start.copy()
    elif obj.ndim == 2 and obj.shape[1] >= 3 and obj.shape[1] % 3 == 0:
        object_points_start = obj[0].reshape(-1, 3)
        object_points_end = obj[-1].reshape(-1, 3)

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection="3d")
frame_idx = np.arange(len(xyz_raw))

ax.plot(xyz_raw[:, 0], xyz_raw[:, 1], xyz_raw[:, 2], color="#2563eb", linewidth=2.0, alpha=0.85, label="EE path")
sc = ax.scatter(
    xyz_raw[:, 0],
    xyz_raw[:, 1],
    xyz_raw[:, 2],
    c=frame_idx,
    cmap="viridis",
    s=11,
    alpha=0.82,
    label="frames",
)
ax.scatter(*xyz_raw[0], s=110, color="#16a34a", marker="o", label="start")
ax.scatter(*xyz_raw[-1], s=130, color="#dc2626", marker="X", label="end")

if len(xyz_raw) > 2:
    step = max(1, len(xyz_raw) // RAW_EE_TRAJECTORY_MAX_ARROWS)
    bases = xyz_raw[:-1:step]
    vecs = np.diff(xyz_raw, axis=0)[::step]
    ax.quiver(
        bases[:, 0], bases[:, 1], bases[:, 2],
        vecs[:, 0], vecs[:, 1], vecs[:, 2],
        length=1.0,
        normalize=False,
        color="#111827",
        alpha=0.45,
        linewidth=0.8,
    )

object_labels = ["cup", "bowl"]
if len(object_points_start):
    for i, p in enumerate(object_points_start):
        label = object_labels[i] if i < len(object_labels) else "object_%d" % i
        ax.scatter(p[0], p[1], p[2], s=95, color="#f97316", marker="^", label="%s start" % label)
        ax.text(p[0], p[1], p[2], "  %s" % label, fontsize=9)
    if len(object_points_end) == len(object_points_start) and np.max(np.abs(object_points_end - object_points_start)) > 1e-6:
        ax.scatter(
            object_points_end[:, 0], object_points_end[:, 1], object_points_end[:, 2],
            s=95,
            color="#7c3aed",
            marker="s",
            label="object end",
        )

set_axes_equal_3d(ax, xyz_raw, object_points_start, object_points_end)
ax.set_xlabel("world x [m]")
ax.set_ylabel("world y [m]")
ax.set_zlabel("world z [m]")
ax.view_init(elev=24, azim=-58)
ax.set_title(
    "Raw EE trajectory: %s\n%s | frames=%d" % (
        selected_npz.stem,
        selected_meta.get("instruction", "<no instruction>"),
        len(xyz_raw),
    )
)
cb = fig.colorbar(sc, ax=ax, shrink=0.68, pad=0.08)
cb.set_label("frame index")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()

RAW_EE_TRAJECTORY_OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(RAW_EE_TRAJECTORY_OUT, dpi=180, bbox_inches="tight")
print("Selected episode:", selected_npz)
print("Saved image:", RAW_EE_TRAJECTORY_OUT)
plt.show()
